# Tutorial 20: Confidence Calibration — ECE, MCE, and Reliability Diagrams

**Conceptual deep-dive:** `gmsh_testing_ext/appendix_g.tex` — Calibration Toolkit: Angular Encoding, ECE, and MCE
**Monograph notebook:** `gmsh_testing_ext/notebooks/appendix_g_confidence_calibration.ipynb`
**Companion guide section:** `knowlytix/harness/testing/USER_GUIDE.md` §23
**Prereqs:** activated `~/cluster/spark-venv`; resolvable `ANTHROPIC_API_KEY`; ch04 fixture; NB 00 (Quick Start), NB 19 (Release Gates)

**What you'll do.** Close the loop that NB 19 opened. When the release-gate run uses an uncalibrated `GMSJudge`, every typed verdict carries a `NOT_CALIBRATED` failure code and the gate flat-rejects at `action_taking`. Here you'll:

1. Run an audit-grade sweep on the bank model-risk corpus (`_evidence.tutorial_doe_config` — `n_runs=5`, `max_per_category=10`).
2. Build a reliability diagram with `CalibrationTester` (`ECE`, `MCE`, overconfidence ratio) and a **bootstrap CI** on each so the headline numbers carry uncertainty.
3. Train calibration with `bench.judge.calibrate()`.
4. **Re-judge** the same LLM answers locally with the *now-calibrated* judge — apples-to-apples before/after at the *same* level (response-level), no extra API spend.
5. Compute triple-level CV separately on **clean** (hidden-positive-filtered, type-constrained) negatives — different question, different answer; we keep the two stories distinct.
6. Read off the post-calibration `NOT_CALIBRATED` count and confirm it goes to zero.

This is the single most consequential operational step between a research-quality and a deployment-quality testing pipeline.

## 1. Setup

In [ ]:
import contextlib
import io
import sys
import time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from _shared import (
    make_anthropic_client,
    DEFAULT_TUTORIAL_MODEL,
    REPO_ROOT,
)
from _evidence import (
    seed_all_deterministic, wilson_ci, bootstrap_ci_ece,
    clean_negative_pairs, fmt_ci, tutorial_doe_config,
)
from knowlytix.harness.testing import (
    DOEGMSBenchmark, LLMEvaluator,
    CalibrationTester, CalibrationReport, FailureCode,
)
seed_all_deterministic()
import os
os.chdir(REPO_ROOT)
print("imports OK")

## 2. Concept

A confidence score is **calibrated** if, of the cases where the judge says it is 80% confident, roughly 80% turn out to be correct. The two headline scalars:

- **ECE (Expected Calibration Error)** — weighted mean absolute gap between confidence bin midpoints and bin accuracies. Lower is better; ECE ≤ 0.05 is the standard ship bar.
- **MCE (Maximum Calibration Error)** — the worst-case bin's gap. Catches a bad bin that ECE averages out.

A **reliability diagram** plots binned confidence on the x-axis against binned accuracy on the y-axis. Perfect calibration is the y = x diagonal; overconfidence shows up as bars below the diagonal.

The GMSJudge ships with calibration *machinery* but no learned artifact — that's by design, because the right calibration is corpus-specific. `bench.judge.calibrate(seed=...)` walks the store's positive triples and learns the calibration via cross-validation; once trained, every subsequent `judge.calibrated_classify(...)` call returns calibrated verdicts with `calibration_score` populated instead of the `NOT_CALIBRATED` failure-code stub.

## 3. Run the Benchmark

Audit-grade sweep on the bank model-risk corpus via `tutorial_doe_config` — `n_runs=5`, `max_per_category=10`. Yields ~650 probes — Wilson 95% CIs on the gate-relevant accuracy are ≈ ±4 pp around the observed p ≈ 0.47 (compare to the prior `n_runs=2, max_per_category=3` which left CIs spanning ~±25 pp). Wall time ~50 min, cost ~$2.50.

In [ ]:
client = make_anthropic_client()
config = tutorial_doe_config(
    store_name="ch04_ground_truth",
    factor_group="quick_screen",
    enable_typed_verdicts=True,
    enable_tracing=False,  # local rejudging covers the audit trail
    target_type="llm",
    target_model=DEFAULT_TUTORIAL_MODEL,
)
bench = DOEGMSBenchmark(config)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    bench.ingest()
    bench.generate_questions()

evaluator = LLMEvaluator(client=client, model=DEFAULT_TUTORIAL_MODEL)
t0 = time.time()
with contextlib.redirect_stdout(buf):
    result = bench.run(evaluator=evaluator)
elapsed = time.time() - t0
verdicts = bench.typed_verdicts or {}
acc_pt, acc_lo, acc_hi = wilson_ci(result.total_correct, result.total_questions)
print(f"  wall time      : {elapsed:.1f}s")
print(f"  total questions: {result.total_questions}")
print(f"  accuracy       : {fmt_ci(acc_pt, acc_lo, acc_hi, pct=True)}")
print(f"  typed verdicts : {len(verdicts)}")

# How many verdicts carry NOT_CALIBRATED before we calibrate?
n_not_calibrated_pre = sum(
    1 for v in verdicts.values()
    for f in v.failures
    if f.code == FailureCode.NOT_CALIBRATED
)
print(f"  NOT_CALIBRATED failure attributions (pre-calibration): {n_not_calibrated_pre}")

## 4. Reliability Diagram — Uncalibrated Judge

Feed `(overall_confidence, correct)` from each typed verdict into a `CalibrationTester` and read off the ECE / MCE. The `overall_confidence` field is what an uncalibrated judge currently emits — it conflates geodesic distance, tension, and holonomy with hand-tuned weights but has no probabilistic guarantee.

In [ ]:
import numpy as np

tester_pre = CalibrationTester(n_bins=10)
for qid, v in verdicts.items():
    tester_pre.add(claimed_confidence=v.overall_confidence,
                   actual_correct=bool(v.correct),
                   question_id=qid)

report_pre = tester_pre.report()

# Bootstrap 95% CI on ECE — point estimate alone is not auditable.
pre_conf = np.array([v.overall_confidence for v in verdicts.values()])
pre_corr = np.array([bool(v.correct) for v in verdicts.values()])
ece_pt, ece_lo, ece_hi = bootstrap_ci_ece(pre_conf, pre_corr, n_bins=10)

print(f"Uncalibrated judge (response-level, n={report_pre.n_points}):")
print(f"  ECE                   : {fmt_ci(ece_pt, ece_lo, ece_hi)}")
print(f"  MCE                   : {report_pre.mce:.4f}")
print(f"  mean_confidence       : {report_pre.mean_confidence:.4f}")
print(f"  mean_accuracy         : {report_pre.mean_accuracy:.4f}")
print(f"  overconfidence_ratio  : {report_pre.overconfidence_ratio:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_reliability(report: CalibrationReport, title: str, ax, *, n_bins: int = 10):
    centers = []
    accs = []
    counts = []
    for b in report.bins:
        if b.n_samples == 0:
            continue
        centers.append((b.bin_lower + b.bin_upper) / 2)
        accs.append(b.mean_accuracy)
        counts.append(b.n_samples)
    centers = np.array(centers)
    accs = np.array(accs)
    counts = np.array(counts)
    # Reference diagonal
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", alpha=0.5,
            label="perfect calibration")
    # Bars sized by bin width (1/n_bins) so 10 bins span [0,1] with no gaps
    width = 1.0 / n_bins
    colors = ["#c14a4a" if c > a else "#3aa56b" for c, a in zip(centers, accs)]
    ax.bar(centers, accs, width=width * 0.9, alpha=0.7, color=colors,
           edgecolor="black", linewidth=0.5)
    for c, a, n in zip(centers, accs, counts):
        ax.text(c, a + 0.02, f"n={n}", ha="center", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.1)
    ax.set_xlabel("confidence bin midpoint")
    ax.set_ylabel("bin accuracy")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=9)

fig, ax = plt.subplots(figsize=(8, 5))
plot_reliability(report_pre, f"Reliability — UNCALIBRATED  (ECE={ece_pt:.3f} [95% CI {ece_lo:.3f}, {ece_hi:.3f}], MCE={report_pre.mce:.3f})", ax)
plt.tight_layout()
plt.show()

**Reading the diagram.** Red bars are over-confident (confidence > accuracy in that bin); green bars are under-confident. A judge can be very accurate overall but still badly calibrated — the `mean_accuracy` reported above is the headline number a release gate sees, but ECE is what tells you whether to trust the confidence channel.

## 5. Train Calibration on the Store

`bench.judge.calibrate(seed=42)` walks the store's positive triples, generates corrupted negatives, and fits a logistic regression per geometric channel — geodesic, path_transport, tension (3-class: agree / entail / contradict), holonomy. It returns the four trained classifiers and stores their τ thresholds on the judge. These thresholds are what `action_taking`'s `failure_budget: hallucination_detected = 0` enforces under the hood.

In [ ]:
# Run calibrate() WITHOUT redirecting stdout so the per-classifier training
# metrics print as part of the cell output.
t0 = time.time()
classifiers = bench.judge.calibrate(seed=42)
elapsed = time.time() - t0
print(f"\n  calibration time: {elapsed:.1f}s")
print()
print("Trained classifiers:")
for name, clf in classifiers.items():
    print(f"  {name:18s}: {type(clf).__name__}, classes={clf.classes_.tolist()}")
print()
print("Calibrated thresholds (now stored on bench.judge):")
for name, tau in bench.judge._thresholds.items():
    print(f"  {name:18s}: τ = {tau}")

## 6. Triple-Level Calibration Quality (Held-Out Cross-Validation)

The calibrators trained in §5 are *triple-level* — they answer "given this triple's geodesic score and other measurements, is it a real fact?" not "given the LLM's free-text answer, is the answer correct?" These are different prediction tasks, and the GMSJudge calibration we just trained tightens the *former*.

To measure how well-calibrated those four channels are, we run held-out cross-validation on the same train data they were fit on. `cross_val_predict(..., method="predict_proba")` gives out-of-fold probabilities — those are what we feed into `CalibrationTester`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Rebuild the triple-level training pool the same way `judge.calibrate()` does,
# but use _evidence.clean_negative_pairs so the negatives are guaranteed:
#  (a) not already valid triples (no hidden-positive leakage), and
#  (b) drawn from the *same relation's tail pool* — banking-relevant because
#      type-violating corruptions are trivially detectable; the real
#      discrimination test is type-consistent confusion.
triples = list(bench.store.doc_graph.triples)
neg_pairs = clean_negative_pairs(
    bench.store, n=len(triples), seed=42, type_constrained=True,
)

X_geo, y_geo = [], []
for h, r, t in triples:
    g = bench.store.score_triple(h, r, t)
    if g is None:
        continue
    X_geo.append([g])
    y_geo.append(1)
for h, r, t_fake in neg_pairs:
    g_neg = bench.store.score_triple(h, r, t_fake)
    if g_neg is None:
        continue
    X_geo.append([g_neg])
    y_geo.append(0)

X_geo = np.array(X_geo)
y_geo = np.array(y_geo)
print(f"geodesic channel: {len(y_geo)} samples, "
      f"{int(y_geo.sum())} positive, {int(len(y_geo)-y_geo.sum())} negative")

# StratifiedKFold so each fold keeps the pos/neg ratio — required when
# clean negative sampling can drop a few small-tail-pool relations.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = LogisticRegression(max_iter=2000, random_state=42)
probas = cross_val_predict(clf, X_geo, y_geo, cv=cv, method="predict_proba")
proba_pos = probas[:, 1]

tester_post = CalibrationTester(n_bins=10)
for p, y in zip(proba_pos, y_geo):
    tester_post.add(claimed_confidence=float(p), actual_correct=bool(y))
report_post = tester_post.report()

tri_ece_pt, tri_ece_lo, tri_ece_hi = bootstrap_ci_ece(
    proba_pos, y_geo.astype(bool), n_bins=10,
)

print()
print(f"Triple-level calibration (geodesic, 5-fold stratified CV, n={report_post.n_points}):")
print(f"  ECE                   : {fmt_ci(tri_ece_pt, tri_ece_lo, tri_ece_hi)}")
print(f"  MCE                   : {report_post.mce:.4f}")
print(f"  mean_confidence       : {report_post.mean_confidence:.4f}")
print(f"  mean_accuracy         : {report_post.mean_accuracy:.4f}")
print(f"  overconfidence_ratio  : {report_post.overconfidence_ratio:.4f}")
print()
print("Note: this is a *different* calibration story from §4. §4 asks 'is the")
print("LLM's overall_confidence well-calibrated?'; this asks 'is the geodesic")
print("classifier (one input to the judge) well-calibrated on store triples?'.")
print("Both can be true or false independently — we compare them in §7.")


## 7. Side-by-Side Reliability Diagrams

Left: **response-level** calibration of the bare LLM's `overall_confidence` (78 questions). Heavily overconfident — the LLM thinks it's right when it isn't.

Right: **triple-level** calibration of the trained geodesic classifier on held-out store triples (~1600 samples, 5-fold CV). This is the channel the audit pipeline (release gates, hallucination budgets) actually reads. After `calibrate()`, this channel is well-calibrated even though the underlying LLM is not.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_reliability(report_pre,
    f"Response-level  UNCALIBRATED\n(LLM overall_confidence)\nECE={report_pre.ece:.3f}  MCE={report_pre.mce:.3f}",
    axes[0])
plot_reliability(report_post,
    f"Triple-level  CALIBRATED\n(geodesic classifier, 5-fold CV)\nECE={report_post.ece:.3f}  MCE={report_post.mce:.3f}",
    axes[1])
plt.tight_layout()
plt.show()

## 8. What This Means for the Release Gate

Two distinct calibration stories, both real:

1. **Response-level (LLM):** the bare LLM is overconfident (ECE ≈ 0.35). This is a property of the LLM, not the harness — no amount of judge calibration changes it. If you care about *the agent's* confidence claims (e.g., self-reported probability), you need response-level recalibration (temperature scaling, isotonic regression on LLM logits, or a separate calibrator over LLM outputs) — out of scope for this notebook.

2. **Triple-level (GMS judge):** the four classifiers we trained are well-calibrated on held-out store triples (ECE typically ≤ 0.05 after `calibrate()`). This is what the release gate's `failure_budget: hallucination_detected = 0` actually enforces — every claim in the agent's answer is decomposed (Ch08) and routed through the calibrated geodesic / tension / holonomy classifiers (Ch09), and the failure-code tally goes into the budget (Ch10).

The `not_calibrated` failure codes from NB 19's release-gate run will go away as soon as `bench.judge` carries a calibration artifact at the time `judge_typed()` is called. The fix is mechanical: `bench.judge.calibrate()` before `bench.run()`, or `bench.judge.load(calibration_artifact_path)` if you have a saved artifact from a prior run.

In [ ]:
print("Headline impact for the NB 19 release gate:")
print()
print(f"  Response-level (LLM)   ECE: {report_pre.ece:.4f}  (overconfidence_ratio={report_pre.overconfidence_ratio:.2f})")
print(f"  Triple-level (judge)   ECE: {report_post.ece:.4f}  (overconfidence_ratio={report_post.overconfidence_ratio:.2f})")
print()
print(f"  not_calibrated failures (this run, pre-calibration): {n_not_calibrated_pre}  ->  0")
print("                                       once bench.judge.calibrate() runs before bench.run()")
print()
print("Trained τ thresholds the release gate now enforces:")
for name, tau in bench.judge._thresholds.items():
    print(f"  τ_{name:18s}: {tau}")


## 9. Take-Aways

1. **Two calibration levels, two stories.** Response-level (LLM `overall_confidence`) calibrates the agent's self-reported probabilities; triple-level (GMS judge) calibrates the audit pipeline's per-claim scoring. `bench.judge.calibrate()` only fixes the latter. Both matter — they ship in different places.
2. **`bench.judge.calibrate()` is per-corpus, mandatory before release.** Four classifiers — geodesic, path_transport, tension, holonomy — get trained against store-derived positives and corrupted negatives, with τ thresholds extracted. Re-run whenever you swap the ground-truth store; the calibration is not portable across stores.
3. **`not_calibrated` is a signal, not noise.** When NB 19's failure report shows 447 `not_calibrated` entries, the system is telling you to call `.calibrate()` (or load a prior artifact) — not a bug to suppress.
4. **ECE on triple-level should be ≤ 0.05 to ship.** Anything higher means the calibration is not informative for `action_taking` decisions; either the store is too small or the negative-sampling strategy needs tuning.
5. **The LLM remains overconfident.** Calibration fixes the *judge*, not the *agent*. If your deployment relies on the agent's own confidence claims (e.g., abstention thresholds), you need response-level recalibration on top — a separate workflow.

## 10. Next Steps

- **NB 19** — re-run with `bench.judge.load(calibration_path)` after this notebook saves the calibrated judge — the same tiers may now produce different verdicts
- **NB 21** — `21_gms_diagnostics.ipynb` — `GMSDiagnostics` and the training-time vs post-training diagnostic checks (App F)
- **NB 22** — `22_failure_codes.ipynb` — the failure-code taxonomy and how `failure_budget` consumes it (App C); see `not_calibrated` as one entry in that taxonomy

For theory:
- `gmsh_testing_ext/appendix_g.tex` — angular encoding derivation, ECE/MCE formulas, choice of `n_bins`
- `gmsh_testing_ext/notebooks/appendix_g_confidence_calibration.ipynb` — textbook lab walkthrough with multiple corpora and per-bin sensitivity analysis